# 3. Collapse Thresholds, Extinction Times and Basins

<div style="border-left:5px solid #16803d;padding:10px"><strong>Question</strong></div>

How much harvesting can the population sustain, which starting populations survive, and how long does collapse take? We build on the equilibria in Sections 1–2. All examples are synthetic, with constant parameters, a well-mixed population and no age structure, delays or environmental randomness.

With $x=N/K$, $\tau=rt$, $a=A/K$, $q=H/(rK)$ and $e=h/r$,
$$
x'=G(x)-q\quad(A),\qquad x'=G(x)-ex\quad(B),\qquad
G(x)=x(1-x)(x/a-1).
$$
Here $x'$ means $dx/d\tau$, and $0<a<1$.

## 3.1 Critical Harvesting Rates

For A, a steady population requires $G(x)=q$, so the largest catch is the maximum of **total growth**. Differentiating gives
$$G'(x)=\frac{-3x^2+2(1+a)x-a}{a}.$$
The larger solution of $G'=0$ is
$$x_* = \frac{1+a+\sqrt{1-a+a^2}}{3},\qquad q_c=G(x_*).$$
It lies between $a$ and $1$ and has $G''(x_*)<0$. In dimensional units, $N_*=Kx_*$ and $H_c=rKq_c$.

For a positive B equilibrium, divide by $x$ to obtain $e=P(x)=G(x)/x$. This is **growth per individual**. Since
$$P'(x)=\frac{1+a-2x}{a},\qquad P''(x)=-2/a<0,$$
its maximum occurs at $(1+a)/2$, giving
$$e_c=\frac{(1-a)^2}{4a},\qquad h_c=re_c=\frac{r(K-A)^2}{4AK}.$$
Below either limit survival is possible, but still depends on the starting population. At equality the positive equilibrium is only one-sided stable. The logistic reference is $H_c=rK/4$; it is a separate model, not an $a\to0$ limit. $H$ has population/time units and $h$ has inverse-time units.

**Algorithm:** evaluate these formulas; independently maximise the expanded total/per-capita growth polynomials; compare the resulting thresholds. Then classify the starting population before calculating an extinction time.

In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

from mm_allee.models import Parameters
from mm_allee.thresholds import dimensional_thresholds

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/wp03.json").exists() and (p / "src/mm_allee").exists()
)
config = json.loads((ROOT / "configs/wp03.json").read_text())
parameters = Parameters(**config["parameters"])
print("Synthetic parameters:", vars(parameters))
print("Dimensional thresholds:", dimensional_thresholds(parameters))

## 3.2 Initial Population and Basins of Attraction

A **basin of attraction** is the set of starting populations that approach the same long-term state. For A with $0<q<q_c$, the signs of $x'$ across the two positive roots are $-,+,-$. Below $x_-$ the population falls; between $x_-$ and $x_+$ it grows; above $x_+$ it falls toward $x_+$. Thus the lower root is an unstable boundary and the upper root is stable.

For B, the corresponding positive roots are
$$x_\pm=\frac{1+a\pm\sqrt{(1-a)^2-4ae}}{2}.$$
The lower is unstable, the upper stable, and zero is always stable because $f'_B(0)=-(1+e)<0$.

| Harvest level | Model A | Model B |
|---|---|---|
| Zero | Below $a$, approaches zero; above $a$, approaches $1$ | Same positive-state basins |
| Positive, below critical | Below $x_-$, reaches zero; above $x_-$, approaches $x_+$ | Below $x_-$, approaches zero; above $x_-$, approaches $x_+$ |
| At critical | Below the merged root, reaches zero; above it, approaches that root | Below the merged root, approaches zero; above it, approaches that root |
| Above critical | Every finite positive start reaches zero | Every positive start approaches zero |

Exact starting equilibria stay there. At the critical harvest the roots merge in a **fold**: arrows point downward on both sides, so the merged root attracts from above but repels from below. This is **semistability**; a zero derivative alone does not establish stability.

For positive constant catch, zero is a stopping boundary, not an equilibrium of the original A equation. The numerical classifier reports non-exact starts within $2\times10^{-12}$ of an unstable/semistable boundary as unresolved. This tolerance describes numerical resolution and does not move the critical harvest.

## 3.3 Time to Extinction

On a declining A path that reaches zero, write $x'=-(q-G(x))$. Separating variables and reversing the limits gives
$$
\tau_{\rm ext}=\int_0^{x_0}\frac{du}{q-G(u)},\qquad
t_{\rm ext}=\frac{\tau_{\rm ext}}{r}=\int_0^{N_0}\frac{dN}{H-g(N)}.
$$
First check that the whole path reaches zero: the denominator must stay positive and contain no equilibrium. For positive subcritical catch below the lower root, or critical catch below the merged root, it is positive and continuous on the interval, so the time is finite. Above the critical catch,
$$q-G(u)\ge q-q_c>0\quad\Rightarrow\quad\tau_{\rm ext}\le\frac{x_0}{q-q_c}.$$

For B, and for A with zero catch, zero is an equilibrium of a smooth ODE. Uniqueness prevents a positive trajectory reaching it in finite time. Approaching a tiny population is therefore different from reaching exact zero.

## 3.4 Numerical Method and Independent Accuracy Checks

<div style="border-left:5px solid #6250d6;padding:10px"><strong>Algorithm</strong></div>

1. Classify the basin analytically; integrate only paths that reach zero.
2. Evaluate the separated integral by adaptive quadrature. Near difficult boundaries, changes of variable or a special fold evaluation preserve accuracy; these evaluate the same integral.
3. Solve the ODE with RK45 at two tolerances, stopping at the first downward zero event. Compare with a separately expanded-polynomial DOP853 solution.
4. Retain every case and its outcome. A missing finite time is not a time of zero, and a short simulation without an event does not prove persistence.

RK45 uses fifth- and fourth-order estimates to adapt its step size. Plotting times do not set its internal steps. The shared reference defaults are $\mathrm{rtol}=10^{-9}$, $\mathrm{atol}=10^{-11}$ and maximum dimensionless step $0.1$.

The six event-validation cases compare quadrature with RK45 at $10^{-8}$ and $10^{-10}$ relative tolerance and DOP853 at $10^{-12}$; absolute tolerances are $10^{-10}$, $10^{-12}$ and $10^{-14}$, respectively. All use maximum step $0.1$. The acceptance condition is
$$|\Delta\tau|\le2\times10^{-7}+5\times10^{-7}\tau_{\rm ext}.$$
This checks the tested cases, not a rigorous global error bound. No negative population or artificial post-extinction zero tail is used as a solution of the original equation. See the official [RK45](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) and [quadrature](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html) documentation.

The code below runs **only the core WP03 analysis** and generates the clean reference data. Optional sensitivity, yield optimisation, policy margins and bottleneck asymptotic comparisons are not executed.

In [ ]:
outputs = {
    "analysis": ROOT / "results/wp03-notebook",
    "reference": ROOT / "results/wp03-reference-notebook",
}
commands = [
    ("analysis", "experiments/wp03/run.py", "configs/wp03.json", ["--core-only"]),
    ("reference", "experiments/wp03/generate_reference.py", "configs/wp03_reference.json", []),
]
for label, script, configuration, extra in commands:
    completed = subprocess.run(
        [
            sys.executable,
            str(ROOT / script),
            "--config",
            str(ROOT / configuration),
            "--output",
            str(outputs[label]),
            *extra,
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())


def read_rows(filename):
    with (outputs["analysis"] / filename).open(newline="") as stream:
        return list(csv.DictReader(stream))


evidence = read_rows("wp03_event_validation.csv")
assert all(row["status"] == "passed" for row in evidence)
print(
    "Largest recorded event difference in dimensionless time:",
    max(float(row["max_abs_error_tau"]) for row in evidence),
)
display(Image(filename=str(outputs["analysis"] / "wp03_basins.png")))
display(Image(filename=str(outputs["analysis"] / "wp03_extinction.png")))

**Interpretation.** For $r=0.7$, $K=100$ and $A=20$, the thresholds are $N_*=70.55050$, $H_c=36.75966$ and $h_c=0.56$; the separate logistic reference is $17.5$.

The 427 curve evaluations retain equilibrium, persistence, asymptotic and unresolved outcomes. Initial zero has time zero and is omitted from logarithmic time axes. For a fixed reaching-zero start, increasing constant catch reduces extinction time; approaching a basin boundary can make it very large.

All six saved event cases passed, with largest discrepancy approximately $4.56\times10^{-8}$ in dimensionless time. The cell prints the result of the current run as well. The basin colour regions meet at the exact critical control; the finite-resolution picture complements the exact-root classification.

## 3.5 Slow Decline Near the Critical Catch

Just above $q_c$, no positive equilibrium remains, but growth almost balances harvesting near the former equilibrium population $x_*$. The net decline is small there, so a trajectory can spend a long time in this region before collapsing. This delay is called a **bottleneck**.

The next figure shows numerical trajectories at three catches just above the critical value. Each ends at its zero event. Catches closer to the threshold produce longer delays, so a short apparently flat trajectory should not be mistaken for permanent survival. This is a visual observation of the computed trajectories; no asymptotic scaling law is fitted or claimed.

In [ ]:
display(Image(filename=str(outputs["analysis"] / "wp03_bottleneck.png")))

## 3.6 Synthetic Reference Data and Limitations

The early handoff contains **six clean synthetic trajectories, 606 sampled points**, covering both models and regular, decline/extinction and near-critical regimes. Their parameters, units and solver settings are recorded. These clean references support the later neural-network packages; they are distinct from WP05's later observation counts.

WP04 owns the forward neural engine. WP05 owns noise generation, frozen train/validation/test splits and inverse fitting. Independently known conditions and fixed external scales remain separate from evaluation truth; hidden unknown parameters and held-out clean values must not enter fitting. The next figure shows the reference trajectories.

<div style="border-left:5px solid #16803d;padding:10px"><strong>Main result</strong></div>

Harvesting limits and initial population jointly determine survival. Positive constant catch can cause finite-time extinction; proportional harvesting can approach zero asymptotically. Near the critical catch, a long delay can precede collapse. Independent numerical comparisons support the tested calculations.

**Limitations.** Constant parameters, a well-mixed population and no environmental/demographic randomness, age structure or delays. These are model consequences, not species-specific recommendations. Reproduction and evidence: `docs/WP03_ANALYSIS.md` and `docs/WP03_CORE_NOTEBOOK.md`. AI assistance for implementation, explanation and checking is recorded in `docs/AI_USAGE.md`; humans review and explain the submitted work.

In [ ]:
display(Image(filename=str(outputs["reference"] / "wp03_rk45_reference.png")))